In [19]:
def parse_gmt(file_path):
    gene_sets = {}
    with open(file_path, 'r', encoding='utf-8') as f:
        for line in f:
            # GMT文件以制表符分隔：通路名 \t 描述 \t 基因1 \t 基因2...
            parts = line.strip().split('\t')
            if len(parts) < 3:
                continue
            
            pathway_name = parts[0]
            # parts[1] 是描述信息（通常是URL），我们可以跳过
            genes = parts[2:]
            gene_sets[pathway_name] = genes
            
    return gene_sets

# 使用示例
gmt_path = "/home/featurize/work/cs-main/DeepTGI/dataset/scRNA-Seq/BP/m5.go.bp.v2026.1.Mm.symbols.gmt"
go_bp_dict = parse_gmt(gmt_path)

# 查看通路数量和第一个通路示例
print(f"共加载了 {len(go_bp_dict)} 条 GO BP 通路")
first_key = list(go_bp_dict.keys())[0]
if "Sertad2" in go_bp_dict["GOBP_REGULATION_OF_CELL_GROWTH"]:
    print(1)

共加载了 7781 条 GO BP 通路
1


In [2]:
import pandas as pd
import numpy as np

import pandas as pd
a=pd.read_csv("ExpressionData.csv")
column_names = list(a.columns)
print(len(column_names))

def gmt_to_binary_matrix_with_orphan_flag(gmt_path, target_genes):
    # 1. 预处理：统一转大写
    target_genes_upper = [str(g).upper() for g in target_genes]
    
    pathway_dict = {}
    with open(gmt_path, 'r', encoding='utf-8') as f:
        for line in f:
            parts = line.strip().split('\t')
            if len(parts) < 3:
                continue
            pathway_name = parts[0]
            # 统一转大写匹配
            genes_in_pathway = set([g.upper() for g in parts[2:]])
            pathway_dict[pathway_name] = genes_in_pathway

    print(f"解析完成，共包含 {len(pathway_dict)} 条通路。")

    # 2. 构建二值矩阵
    matrix_data = []
    pathway_names = []

    for name, genes in pathway_dict.items():
        row = [1 if g in genes else 0 for g in target_genes_upper]
        matrix_data.append(row)
        pathway_names.append(name)

    # 先构建初步的 DataFrame
    adj_matrix = pd.DataFrame(
        matrix_data, 
        index=pathway_names, 
        columns=target_genes,
        dtype=np.int8
    )

    # 3. 核心逻辑：增加一个维度标识“无通路基因”
    # 检查哪些列（基因）的和等于 0
    # sum(axis=0) 对列求和，得到每个基因参与的通路总数
    gene_sums = adj_matrix.sum(axis=0)
    
    # 定义新的一行：如果没有被任何通路覆盖（sum == 0），则置为 1，否则为 0
    orphan_row = (gene_sums == 0).astype(np.int8)
    
    # 将这一行命名为 'NO_ANNOTATED_GO_BP' 并添加到矩阵底部
    adj_matrix.loc['NO_ANNOTATED_GO'] = orphan_row

    return adj_matrix

# --- 执行 ---
gmt_file = "/home/featurize/work/cs-main/DeepTGI/dataset/scRNA-Seq/BP/m5.go.cc.v2026.1.Mm.symbols.gmt"
my_genes = column_names

adj_df = gmt_to_binary_matrix_with_orphan_flag(gmt_file, my_genes)

# 验证
orphan_count = adj_df.loc['NO_ANNOTATED_GO'].sum()
print(f"处理完成！共有 {orphan_count} 个基因没有任何 GO 注释，已标记在 'NO_ANNOTATED_GO' 维度。")

# 保存结果
# 建议保留 index，因为最后一行 'NO_ANNOTATED_GO_BP' 是一个重要的特征维度
adj_df.to_csv("mDC_CC.csv",index=False)

1321
解析完成，共包含 1067 条通路。
处理完成！共有 276 个基因没有任何 GO 注释，已标记在 'NO_ANNOTATED_GO' 维度。


In [3]:
#work/cs-main/DeepTGI/dataset/scRNA-Seq/BP/m5.go.bp.v2026.1.Mm.symbols.gmt
import pandas as pd
a=pd.read_csv("mDC_BP2.csv")
del a["Unnamed: 0"]
a.to_csv("mDC_BP1.csv",index=False)